##Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, StandardScaler

##Load Datasets

In [ ]:
high = pd.read_csv("/content/maternal_high_burden.csv")
moderate = pd.read_csv("/content/maternal_moderate_burden.csv")
low = pd.read_csv("/content/maternal_low_burden.csv")

In [ ]:
print(high.columns)
print(moderate.columns)
print(low.columns)

##Merge Datasets

In [ ]:
import pandas as pd

high = pd.read_csv("/content/maternal_high_burden.csv")
moderate = pd.read_csv("/content/maternal_moderate_burden.csv")
low = pd.read_csv("/content/maternal_low_burden.csv")

df = pd.concat([high, moderate, low], ignore_index=True)

print(df.shape)
df.head()

In [ ]:
df.to_csv("maternal_health_combined.csv", index=False)

##Data Understanding

In [ ]:
df.shape
df.info()
df.isnull().sum()

##Data Cleaning

In [ ]:
df.drop("id", axis=1, inplace=True)

##Risk Level Distribution

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

sns.countplot(x="risk_level", data=df)

plt.title("Distribution of Pregnancy Risk Levels")
plt.show()

In [ ]:
risk_counts = df['risk_level'].value_counts()

plt.figure(figsize=(6,6))

plt.pie(
risk_counts,
labels=risk_counts.index,
autopct='%1.1f%%'
)

plt.title("Risk Level Percentage")

plt.show()


##Numerical Feature Distributions

In [ ]:
numeric_cols = df.select_dtypes(include='number').columns

for col in numeric_cols:
    plt.figure()
    sns.histplot(df[col], kde=True)
    plt.title(col + " Distribution")
    plt.show()

##Scatter Plots

In [ ]:
plt.figure(figsize=(7,5))

sns.scatterplot(
x='bmi_pre_pregnancy',
y='fasting_glucose_mgdl',
hue='risk_level',
data=df
)

plt.title("BMI vs Glucose by Risk Level")
plt.show()

In [ ]:
sns.scatterplot(
x='age_years',
y='bmi_pre_pregnancy',
hue='risk_level',
data=df
)

plt.title("Age vs BMI")
plt.show()

##Feature Analysis

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

numeric_features = [
    'age_years',
    'gravidity',
    'parity',
    'gestational_age_weeks',
    'bmi_pre_pregnancy',
    'systolic_bp_mmhg',
    'diastolic_bp_mmhg',
    'hemoglobin_gdl',
    'fasting_glucose_mgdl',
    'anc_visits'
]

categorical_features = [
    'proteinuria',
    'hiv_status'
]

##Numerical Features vs Risk Level

In [ ]:
for feature in numeric_features:
    plt.figure(figsize=(8,5))
    sns.barplot(x='risk_level', y=feature, data=df, errorbar=None)
    plt.title(f"Average {feature} by Risk Level")
    plt.xlabel("Risk Level")
    plt.ylabel(feature)
    plt.tight_layout()
    plt.show()

##Categorical Features vs Risk Level

In [ ]:
for feature in categorical_features:
    plt.figure(figsize=(8,5))
    sns.countplot(x=feature, hue='risk_level', data=df)
    plt.title(f"Distribution of {feature} by Risk Level")
    plt.xlabel(feature)
    plt.ylabel("Count")
    plt.legend(title='Risk Level')
    plt.tight_layout()
    plt.show()

##Outlier Detection

In [ ]:
for col in numeric_features:
    sns.boxplot(x=df[col])
    plt.show()


##Correlation Analysis

In [ ]:
plt.figure(figsize=(12, 10))

correlation_matrix = df.corr(numeric_only=True)

sns.heatmap(
    correlation_matrix,
    annot=True,
    cmap='coolwarm',
    fmt=".2f",
    linewidths=0.5,
    vmin=-1,
    vmax=1
)

plt.title("Correlation Matrix of Maternal Health Features")
plt.tight_layout()
plt.show()


##Feature Engineering

In [ ]:
# Feature Engineering

df['bp_diff'] = (
    df['systolic_bp_mmhg']
    - df['diastolic_bp_mmhg']
)

df.head()

##Remove Unneeded Columns

In [ ]:
df.drop(
    [
        'delivery_mode',
        'pregnancy_outcome',
        'primary_complication'
    ],
    axis=1,
    inplace=True
)

##Outlier Treatment

In [ ]:
numeric_cols = [
    'age_years',
    'gravidity',
    'parity',
    'gestational_age_weeks',
    'bmi_pre_pregnancy',
    'systolic_bp_mmhg',
    'diastolic_bp_mmhg',
    'hemoglobin_gdl',
    'fasting_glucose_mgdl',
    'anc_visits',
    'bp_diff'
]

for col in numeric_cols:

    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)

    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    df[col] = df[col].clip(lower, upper)

for col in numeric_cols:
    sns.boxplot(x=df[col])
    plt.show()


##Encoding

In [ ]:
df['anemia_status'] = df['anemia_status'].map({
    'none': 0,
    'mild': 1,
    'moderate': 2,
    'severe': 3
})
df['risk_level'] = df['risk_level'].map({
    'low': 0,
    'moderate': 1,
    'high': 2
})

##Feature & Target Selection

In [ ]:
X = df.drop('risk_level', axis=1)
y = df['risk_level']

##Train Test Split

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

##Feature Scaling

In [ ]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

##Random Forest Training

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=200,
    random_state=42
)

rf_model.fit(X_train, y_train)
y_pred = rf_model.predict(X_test)

##Model Evaluation

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

##Confusion Matrix

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred)

sns.heatmap(cm, annot=True, fmt='d')
plt.title("Confusion Matrix")
plt.show()

##Feature Importance

In [ ]:
import pandas as pd

importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': rf_model.feature_importances_
})

importance = importance.sort_values(by='Importance', ascending=False)

print(importance)
plt.figure(figsize=(8,5))
sns.barplot(data=importance, x='Importance', y='Feature')
plt.title("Feature Importance")
plt.show()

##Save Model

In [ ]:
import joblib

joblib.dump(rf_model, "maternal_risk_model.pkl")
joblib.dump(scaler, "scaler.pkl")

##Class Distribution After Encoding

In [ ]:
print(df['anemia_status'].value_counts())
print(df['risk_level'].value_counts())

##Cross Validation

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import RandomForestClassifier

rf_cv = RandomForestClassifier(n_estimators=200, random_state=42)

pipeline = make_pipeline(StandardScaler(), rf_cv)

scores = cross_val_score(pipeline, X, y, cv=5, scoring='accuracy')

print("Scores:", scores)
print("Mean Accuracy:", scores.mean())
print("Std:", scores.std())

##Model Comparison

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import pandas as pd

lr_model = LogisticRegression(max_iter=1000)
dt_model = DecisionTreeClassifier()

results = {}

# Logistic Regression (needs scaling)
lr_model.fit(X_train_scaled, y_train)
lr_pred = lr_model.predict(X_test_scaled)
results["Logistic Regression"] = accuracy_score(y_test, lr_pred)

# Decision Tree (no scaling)
dt_model.fit(X_train, y_train)
dt_pred = dt_model.predict(X_test)
results["Decision Tree"] = accuracy_score(y_test, dt_pred)

# Random Forest (no scaling)
rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)
results["Random Forest"] = accuracy_score(y_test, rf_pred)

results_df = pd.DataFrame(results.items(), columns=["Model", "Accuracy"])
print(results_df)

##Model Comparison Visualization

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(6,4))
sns.barplot(data=results_df, x="Model", y="Accuracy")
plt.title("Model Comparison")
plt.xticks(rotation=20)
plt.show()

# Best Model Selection

In [ ]:
best_model = results_df.loc[
    results_df["Accuracy"].idxmax()
]

print("Best Model:")
print(best_model)

# Prediction Probability

In [ ]:
sample = X_test.iloc[[0]]

prediction = rf_model.predict(sample)

probabilities = rf_model.predict_proba(sample)

print("Prediction:", prediction[0])
print("Probabilities:", probabilities)

# Risk Confidence Score

In [ ]:
confidence = probabilities.max() * 100

print(f"Confidence Score: {confidence:.2f}%")

# Explainable AI Using SHAP

In [ ]:
import shap
shap.initjs()

explainer = shap.TreeExplainer(rf_model)
sample_data = X_test.iloc[:100]

shap_values = explainer.shap_values(sample_data)

shap.summary_plot(
    shap_values,
    sample_data
)

# Top Risk Factors Analysis

In [ ]:
feature_importance = pd.DataFrame({
    "Feature": X_test.columns,
    "Importance": abs(
        shap_values[0, :, prediction[0]]
    )
})

top_features = (
    feature_importance
    .sort_values(
        "Importance",
        ascending=False
    )
    .head(5)
)

top_features

# Explain Individual Prediction

In [ ]:
print(type(shap_values))
print(len(shap_values))

print(shap_values.shape)
print(sample.shape)

In [ ]:
sample = X_test.iloc[[0]]

pred_class = prediction[0]

shap.force_plot(
    explainer.expected_value[pred_class],
    shap_values[0, :, pred_class],
    sample.iloc[0],
    matplotlib=True
)

# Pregnancy Risk Score

In [ ]:
risk_score = probabilities.max() * 100

print(f"Risk Score = {risk_score:.0f}/100")

# Recommendation Engine

In [ ]:
pred = rf_model.predict(sample)[0]

if pred == 0:
    recommendation = """
    Low Risk Pregnancy

    Continue routine checkups.
    Maintain healthy nutrition.
    Stay physically active.
    """

elif pred == 1:
    recommendation = """
    Moderate Risk Pregnancy

    Increase monitoring frequency.
    Track blood pressure regularly.
    Follow doctor's instructions.
    """

else:
    recommendation = """
    High Risk Pregnancy

    Immediate medical consultation.
    Daily blood pressure monitoring.
    Frequent fetal assessment.
    """

print(recommendation)

# Emergency Alert System

In [ ]:
if risk_score > 90:
    print("🚨 EMERGENCY ALERT")
    print("Immediate medical attention is recommended.")

# AI Explanation System

In [ ]:
feature_importance = pd.DataFrame(
    {
        "Feature": X.columns,
        "Importance": rf_model.feature_importances_
    }
)

top_features = feature_importance.sort_values(
    by="Importance",
    ascending=False
).head(3)

print("Main Risk Factors:")

for feature in top_features["Feature"]:
    print("-", feature)

In [ ]:
import shap

shap.initjs()

sample = X_test.iloc[[0]]
prediction = rf_model.predict(sample)

# Calculate SHAP values specifically for the single instance
shap_values_for_instance = explainer.shap_values(sample)

shap.force_plot(
    explainer.expected_value[prediction[0]],
    shap_values_for_instance[0, :, prediction[0]],
    sample.values[0]
)

#AI Generated Medical Summary

In [ ]:
feature_importance = pd.DataFrame(
    {
        "Feature": X.columns,
        "Importance": rf_model.feature_importances_
    }
)

top_features = feature_importance.sort_values(
    by="Importance",
    ascending=False
).head(3)

print("Main Risk Factors:")

for feature in top_features["Feature"]:
    print("-", feature)

# Personalized Pregnancy Health Tips Generator

In [ ]:
if risk_score < 40:

    tips = [
        "Continue routine prenatal checkups.",
        "Maintain a balanced diet rich in nutrients.",
        "Stay physically active with light exercise.",
        "Drink enough water daily.",
        "Take prenatal vitamins as prescribed."
    ]

elif risk_score < 70:

    tips = [
        "Increase the frequency of prenatal monitoring.",
        "Track blood pressure regularly.",
        "Maintain healthy weight gain.",
        "Get adequate sleep and rest.",
        "Report any unusual symptoms to your doctor."
    ]

else:

    tips = [
        "Consult your healthcare provider immediately.",
        "Monitor blood pressure daily.",
        "Track fetal movements carefully.",
        "Avoid excessive physical stress.",
        "Attend all scheduled prenatal appointments.",
        "Follow a nutrition plan recommended by your doctor."
    ]

print("PERSONALIZED PREGNANCY HEALTH TIPS\n")

for i, tip in enumerate(tips, start=1):
    print(f"{i}. {tip}")

# PDF Medical Report Generation

In [ ]:
!pip install reportlab

In [ ]:
from reportlab.platypus import (
    SimpleDocTemplate,
    Paragraph,
    Spacer
)

from reportlab.lib.styles import getSampleStyleSheet

# -----------------------------------
# Generate Personalized Tips
# -----------------------------------

if risk_score < 40:

    tips = [
        "Continue routine prenatal checkups.",
        "Maintain a balanced diet rich in nutrients.",
        "Stay physically active with light exercise.",
        "Drink enough water daily.",
        "Take prenatal vitamins as prescribed."
    ]

elif risk_score < 70:

    tips = [
        "Increase the frequency of prenatal monitoring.",
        "Track blood pressure regularly.",
        "Maintain healthy weight gain.",
        "Get adequate sleep and rest.",
        "Report unusual symptoms to your doctor."
    ]

else:

    tips = [
        "Consult your healthcare provider immediately.",
        "Monitor blood pressure daily.",
        "Track fetal movements carefully.",
        "Avoid excessive physical stress.",
        "Attend all scheduled prenatal appointments.",
        "Follow a nutrition plan recommended by your doctor."
    ]

# -----------------------------------
# Create PDF
# -----------------------------------

report = SimpleDocTemplate(
    "Pregnancy_Risk_Report.pdf"
)

styles = getSampleStyleSheet()

content = []

# Title

content.append(
    Paragraph(
        "Pregnancy Risk Assessment Report",
        styles["Title"]
    )
)

content.append(Spacer(1,12))

# Risk Level

content.append(
    Paragraph(
        f"<b>Risk Level:</b> {prediction[0]}",
        styles["BodyText"]
    )
)

# Risk Score

content.append(
    Paragraph(
        f"<b>Risk Score:</b> {risk_score}/100",
        styles["BodyText"]
    )
)

content.append(Spacer(1,12))

# Risk Factors

content.append(
    Paragraph(
        "Top Risk Factors",
        styles["Heading2"]
    )
)

for feature in top_features["Feature"]:

    content.append(
        Paragraph(
            f"• {feature}",
            styles["BodyText"]
        )
    )

content.append(Spacer(1,12))

# Recommendations

content.append(
    Paragraph(
        "Recommendations",
        styles["Heading2"]
    )
)

if risk_score < 40:

    recommendations = [
        "Continue routine prenatal visits.",
        "Maintain a healthy lifestyle.",
        "Follow a balanced diet."
    ]

elif risk_score < 70:

    recommendations = [
        "Monitor health regularly.",
        "Increase prenatal follow-up.",
        "Track blood pressure frequently."
    ]

else:

    recommendations = [
        "Consult a healthcare provider immediately.",
        "Monitor symptoms closely.",
        "Perform frequent maternal and fetal assessments."
    ]

for rec in recommendations:

    content.append(
        Paragraph(
            f"• {rec}",
            styles["BodyText"]
        )
    )

content.append(Spacer(1,12))

# Personalized Tips

content.append(
    Paragraph(
        "Personalized Pregnancy Health Tips",
        styles["Heading2"]
    )
)

for tip in tips:

    content.append(
        Paragraph(
            f"• {tip}",
            styles["BodyText"]
        )
    )

content.append(Spacer(1,12))

# Medical Summary

content.append(
    Paragraph(
        "Medical Summary",
        styles["Heading2"]
    )
)

summary_text = f"""
The patient is classified as {prediction[0]}
with a risk score of {risk_score}/100.

The most influential factors are:
{', '.join(top_features['Feature'].tolist())}.

The patient should follow the provided recommendations
and maintain regular prenatal follow-up.
"""

content.append(
    Paragraph(
        summary_text,
        styles["BodyText"]
    )
)

# Build PDF

report.build(content)

print("Pregnancy_Risk_Report.pdf generated successfully!")

In [ ]:
import os

os.listdir()

# Pregnancy History Tracking

In [ ]:
import pandas as pd
from datetime import datetime
import os

record = {
    "Date": datetime.now(),
    "Risk_Level": prediction[0],
    "Risk_Score": risk_score
}

history_df = pd.DataFrame([record])

if os.path.exists("pregnancy_history.csv"):

    history_df.to_csv(
        "pregnancy_history.csv",
        mode="a",
        header=False,
        index=False
    )

else:

    history_df.to_csv(
        "pregnancy_history.csv",
        index=False
    )

print("Pregnancy history saved successfully.")

In [ ]:
history = pd.read_csv(
    "pregnancy_history.csv"
)

history.tail()

# Pregnancy Risk Trend Analysis

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

history = pd.read_csv(
    "pregnancy_history.csv"
)

history["Date"] = pd.to_datetime(
    history["Date"]
)

In [ ]:
plt.figure(figsize=(10,5))

plt.plot(
    history["Date"],
    history["Risk_Score"],
    marker="o"
)

plt.title(
    "Pregnancy Risk Trend Over Time"
)

plt.xlabel(
    "Date"
)

plt.ylabel(
    "Risk Score"
)

plt.grid(True)

plt.show()

# Historical Prediction Storage

In this section, every prediction generated by the model is stored with its corresponding
date, predicted risk level, probability score, and patient information.

This allows tracking pregnancy history and generating historical reports later in the application.

In [ ]:
from datetime import datetime
import pandas as pd
import os

history = {
    "Date": datetime.now().strftime("%Y-%m-%d %H:%M"),
    "Risk_Level": prediction[0],
    "Risk_Score": round(risk_score,2),
    "Probability": round(probabilities.max()*100,2)
}

history_df = pd.DataFrame([history])

if os.path.exists("prediction_history.csv"):
    history_df.to_csv(
        "prediction_history.csv",
        mode="a",
        header=False,
        index=False
    )
else:
    history_df.to_csv(
        "prediction_history.csv",
        index=False
    )

# Patient Registration Dataset

This section creates a structured patient database that stores demographic
and medical history information for future prediction and follow-up.

In [ ]:
patient_columns = [

    "Full_Name",
    "Age",
    "Phone",
    "Email",
    "Previous_Pregnancy",
    "Previous_Miscarriage",
    "Diabetes",
    "Hypertension",
    "Heart_Disease",
    "Kidney_Disease",
    "Disability",
    "Current_Medications",
    "Allergies"

]

patient_database = pd.DataFrame(columns=patient_columns)

patient_database.head()

In [ ]:
patient_database.head()

patient_database.info()

# AI Medical Report Analyzer

Optical Character Recognition (OCR) is used to automatically extract
medical information from uploaded laboratory reports and clinical documents.

This reduces manual data entry and improves usability.

In [ ]:
# Uncomment if running for the first time

# !pip install pytesseract
!sudo apt install

In [ ]:
import cv2
import pytesseract

image = cv2.imread("sample_report.jpg")

if image is None:
    print("Error: sample_report.jpg not found. Please upload the image file.")
else:
    gray = cv2.cvtColor(image,cv2.COLOR_BGR2GRAY)

    text = pytesseract.image_to_string(gray)

    print(text)

# Personalized Medical Recommendation Engine

Generate personalized recommendations according to the predicted pregnancy
risk level and the patient's clinical measurements.

# Pregnancy Knowledge Base

A structured medical knowledge base containing frequently asked pregnancy
questions categorized by healthcare topic.

In [ ]:
knowledge_base = {

"Nutrition":[
"Eat iron-rich foods.",
"Drink enough water.",
"Take folic acid daily."
],

"Exercise":[
"Walk 30 minutes daily.",
"Avoid heavy lifting."
],

"Warning Signs":[
"Severe bleeding.",
"Persistent headache.",
"Blurred vision."
],

"Medication":[
"Never take medication without consulting a physician."
]

}

knowledge_base

# Rule-Based AI Assistant

A lightweight AI assistant that answers common pregnancy-related questions
using the medical knowledge base.

In [ ]:
def medical_chatbot(question):

    q = question.lower()

    if "nutrition" in q:
        return knowledge_base["Nutrition"]

    elif "exercise" in q:
        return knowledge_base["Exercise"]

    elif "warning" in q:
        return knowledge_base["Warning Signs"]

    elif "medicine" in q:
        return knowledge_base["Medication"]

    else:
        return "Please consult your healthcare provider."

medical_chatbot("nutrition")